In [ ]:
# !pip install -r requirements.txt


In [2]:
import os
import shutil
from google.colab import drive

# 1. Mount Drive (Essential for accessing your saved data.zip)
drive.mount('/content/drive')

# 2. Define your source and destination
# Use the exact path we used for the download earlier
SOURCE_ZIP = "/content/drive/MyDrive/Colab Notebooks/data/data.zip"
DESTINATION_DIR = "/content/dataset"

# 3. Create the local directory
os.makedirs(DESTINATION_DIR, exist_ok=True)

print("Starting Transfer from Drive to Local SSD (this is fast)...")

project_path = f"/content/drive/MyDrive/Colab Notebooks/adl_cv"
os.chdir(project_path)

print("Current Working Directory:", os.getcwd())

Mounted at /content/drive
Starting Transfer from Drive to Local SSD (this is fast)...
Current Working Directory: /content/drive/MyDrive/Colab Notebooks/adl_cv


In [3]:
# !git fetch origin
# !git pull origin medial_sota

In [4]:


# 4. Copy the zip file (using !cp for high-speed Linux transfer)
!cp "{SOURCE_ZIP}" /content/data_local.zip

print("Transfer complete. Starting extraction (this may take 15-20 mins)...")

# 5. Unzip the file into the local folder
# -q flag (quiet) is CRITICAL here; printing 112,000 filenames will crash your VS Code tab.
!unzip -q /content/data_local.zip -d "{DESTINATION_DIR}"

# 6. Clean up the local zip to save space (now that it's extracted)
os.remove("/content/data_local.zip")

print(f"Done! Your {len(os.listdir(DESTINATION_DIR))} images are ready at {DESTINATION_DIR}")

Transfer complete. Starting extraction (this may take 15-20 mins)...
Done! Your 20 images are ready at /content/dataset


In [5]:
# !git pull


In [6]:
!pip install -r requirements.txt
!pip install transformers shap lime pytorch-gradcam

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.2/114.2 kB 8.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 kB 5.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 8.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.0/6.0 MB 105.2 MB/s eta 0:00:00a 0:00:01
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 6.9 MB/s eta 0:00:00
ERROR: Ignored the following versions that require a different python version: 1.10.0 Requires-Python <3.12,>=3.8; 1.10.0rc1 Requires-Python <3.12,>=3.8; 1.10.0rc2 Requires-Python <3.12,>=3.8; 1.10.1 Requires-Python <3.12,>=3.8; 1.21.2 Requires-Python >=3.7,<3.11; 1.21.3 Requires-Python >=3.7,<3.11; 1.21.4 Requires-Python >=3.7,<3.11; 1.21.5 Requires-Python >=3.7,<3.11; 1.21.6 Requires-Python >=3.7,<3.11; 1.6.2 Requires-Python >=3.7,<3.10; 1.6.3 Requires-Python >=3.7,<3.10; 1.7.0 Requires-Python >=3.7,<3.10; 1.7.1 Requires-Python >=3.7,<3.10; 

In [1]:
%%writefile src/config.py
import os

ROOT_DIR = os.path.abspath(os.path.join(os.path.dirname(__file__), ".."))

DATASET_DIR = "/content/dataset"
CSV_PATH = os.path.join(DATASET_DIR, "Data_Entry_2017.csv")

IMAGE_SIZE = 224
NUM_CLASSES = 14
BATCH_SIZE = 128  # if OOM on GPU, lower this value (e.g. 4).
EPOCHS = 10
LR = 5e-4

MODEL_NAME = "chexfound"

# --- Checkpoint Settings ---
# Directory to save model checkpoints, relative to the project root in your Drive.
CHECKPOINT_DIR = os.path.join(ROOT_DIR, "checkpoints")

# To resume training, change this to the path of a checkpoint file.
# e.g., os.path.join(CHECKPOINT_DIR, "latest_checkpoint.pth")
# Set to None to train from scratch.
RESUME_CHECKPOINT_PATH = None

Writing src/config.py


FileNotFoundError: [Errno 2] No such file or directory: 'src/config.py'

In [8]:
# %%writefile src/main.py
# import sys
# import os

# # Adds 'chestxray-classification' (repo root) and 'chestxray-classification/src' to the system path
# # so local imports like `from models...` work when running this file directly.
# root_path = os.path.abspath(os.path.join(os.path.dirname(__file__), '..'))
# src_path = os.path.join(root_path, 'src')
# for p in (root_path, src_path):
#     if p not in sys.path:
#         sys.path.insert(0, p)

# import torch
# import pandas as pd
# from torch.utils.data import DataLoader
# import matplotlib.pyplot as plt

# from build_path_map import build_dataframe_with_paths
# from dataset_loader import NIHDataset
# from data_split import split_data
# from config import *
# from train import train_one_epoch
# from validate import validate_one_epoch
# from model import get_model

# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# df = build_dataframe_with_paths()
# train_df, val_df, test_df = split_data(df)

# train_dataset = NIHDataset(train_df)
# val_dataset = NIHDataset(val_df)

# train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True, prefetch_factor=2)
# val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, num_workers=4, pin_memory=True, prefetch_factor=2)

# model = get_model().to(device)

# criterion = torch.nn.BCEWithLogitsLoss()
# optimizer = torch.optim.Adam(model.parameters(), lr=LR)

# # --- Early Stopping and Checkpoint Variables ---
# start_epoch = 0
# epoch_train_losses, epoch_val_losses = [], []
# epoch_train_accuracies, epoch_val_accuracies = [], []
# best_val_loss = float('inf')
# patience_counter = 0
# EARLY_STOPPING_PATIENCE = 5  # Stop after 5 epochs with no improvement

# # Directory to save model checkpoints
# checkpoint_dir = os.path.join(root_path, "checkpoints")
# os.makedirs(checkpoint_dir, exist_ok=True)
# best_model_path = os.path.join(checkpoint_dir, f"{MODEL_NAME}_best_model.pth")

# # --- Resume Training from Checkpoint ---
# # To resume, set this to the path of the 'latest_checkpoint.pth' file.
# # You can easily change this value right here in the notebook!
# RESUME_CHECKPOINT_PATH = "checkpoints/radjepa_epoch_5.pth" 
# if RESUME_CHECKPOINT_PATH and os.path.exists(RESUME_CHECKPOINT_PATH):
#     print(f"Resuming training from checkpoint: {RESUME_CHECKPOINT_PATH}")
#     checkpoint = torch.load(RESUME_CHECKPOINT_PATH, map_location=device)
    
#     # Check if the checkpoint is in the new comprehensive format
#     if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
#         model.load_state_dict(checkpoint['model_state_dict'])
#         optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        
#         start_epoch = 5 # Start from the next epoch
#         best_val_loss = checkpoint.get('best_val_loss', float('inf'))
        
#         # Restore history
#         epoch_train_losses = checkpoint.get('train_losses', [])
#         epoch_val_losses = checkpoint.get('val_losses', [])
#         epoch_train_accuracies = checkpoint.get('train_accuracies', [])
#         epoch_val_accuracies = checkpoint.get('val_accuracies', [])
        
#         print(f"Resumed from epoch {start_epoch}. Best validation loss: {best_val_loss:.4f}")
#     else:
#         print("Older checkpoint format detected (model weights only). Starting from epoch 0.")
#         model.load_state_dict(checkpoint)

# for epoch in range(start_epoch, EPOCHS):
#     # --- Training ---
#     train_loss, train_acc = train_one_epoch(
#         model, train_loader, optimizer, criterion, device
#     )
    
#     # --- Validation ---
#     val_loss, val_acc = validate_one_epoch(
#         model, val_loader, criterion, device
#     )
    
#     print(
#         f"Epoch {epoch+1}/{EPOCHS} | "
#         f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
#         f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}"
#     )

#     epoch_train_losses.append(train_loss)
#     epoch_val_losses.append(val_loss)
#     epoch_train_accuracies.append(train_acc)
#     epoch_val_accuracies.append(val_acc)

#     # --- Checkpoint and Early Stopping Logic ---
#     # Save the latest state of the training for resumption
#     latest_checkpoint_path = os.path.join(checkpoint_dir, "latest_checkpoint.pth")
#     torch.save({
#         'epoch': epoch,
#         'model_state_dict': model.state_dict(),
#         'optimizer_state_dict': optimizer.state_dict(),
#         'best_val_loss': best_val_loss,
#         'train_losses': epoch_train_losses,
#         'val_losses': epoch_val_losses,
#         'train_accuracies': epoch_train_accuracies,
#         'val_accuracies': epoch_val_accuracies,
#     }, latest_checkpoint_path)

#     # Early stopping based on validation loss
#     if val_loss < best_val_loss:
#         best_val_loss = val_loss
#         patience_counter = 0
#         # Save the best model weights separately
#         torch.save(model.state_dict(), best_model_path)
#         print(f"Validation loss improved. Saved best model to {best_model_path}")
#     else:
#         patience_counter += 1
#         print(f"Validation loss did not improve. Patience: {patience_counter}/{EARLY_STOPPING_PATIENCE}")

#     if patience_counter >= EARLY_STOPPING_PATIENCE:
#         print("Early stopping triggered.")
#         break

# # --- Visualization Code ---
# epochs_ran = len(epoch_train_losses)
# fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# ax1.plot(range(1, epochs_ran + 1), epoch_train_losses, marker='o', linestyle='-', color='b', label='Training Loss')
# ax1.plot(range(1, epochs_ran + 1), epoch_val_losses, marker='o', linestyle='-', color='r', label='Validation Loss')
# ax1.set_title(f'Loss per Epoch ({MODEL_NAME})')
# ax1.set_xlabel('Epoch')
# ax1.set_ylabel('Loss')
# ax1.set_xticks(range(1, epochs_ran + 1))
# ax1.grid(True)
# ax1.legend()

# ax2.plot(range(1, epochs_ran + 1), epoch_train_accuracies, marker='o', linestyle='-', color='b', label='Training Accuracy')
# ax2.plot(range(1, epochs_ran + 1), epoch_val_accuracies, marker='o', linestyle='-', color='r', label='Validation Accuracy')
# ax2.set_title(f'Accuracy per Epoch ({MODEL_NAME})')
# ax2.set_xlabel('Epoch')
# ax2.set_ylabel('Accuracy (Exact Match Ratio)')
# ax2.set_xticks(range(1, epochs_ran + 1))
# ax2.grid(True)
# ax2.legend()

# plt.tight_layout()

# # Save the plot as an image and display it
# plt.savefig(f'training_curves_{MODEL_NAME}.png')
# plt.show()

# print("Training complete.")
# if patience_counter >= EARLY_STOPPING_PATIENCE:
#     print(f"Best model (epoch {epochs_ran - EARLY_STOPPING_PATIENCE}) saved at {best_model_path}")
# else:
#     print(f"Best model saved at {best_model_path}")


In [9]:
# print(xxxxxxxxxxx)

In [10]:
!python src/main.py

Missing image paths: 0/112120
config.json: 100% 215/215 [00:00<00:00, 861kB/s]
modeling_radjepa.py: 2.40kB [00:00, 2.89MB/s]
A new version of the following files was downloaded from https://huggingface.co/AIDElab-IITBombay/RadJEPA:
- modeling_radjepa.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
jepa_encoder.pth.tar: 100% 1.63G/1.63G [00:21<00:00, 77.3MB/s]
Training:  99% 608/614 [08:17<00:03,  1.95it/s]libpng warning: iCCP: profile 'ICC Profile': 'GRAY': Gray color space not permitted on RGB PNG
Training: 100% 614/614 [08:22<00:00,  1.22it/s]
Validating: 100% 131/131 [01:51<00:00,  1.18it/s]
Epoch 1/10 | LR: 0.000500 | Train Loss: 0.1816 | Train Acc: 0.5364 | Val Loss: 0.1760 | Val Acc: 0.5379
Validation loss improved. Saved best model to /content/drive/MyDrive/Colab Notebooks/adl_cv/checkpoints/radjepa_best_model.pth
Training:  56% 344/614 [04:42<02:40,  1.68it/s]libpng warning:

In [11]:
import torch
import os
from src.model import get_model
from src.config import MODEL_NAME

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = get_model()

# Define the path to your saved checkpoint (adjust epoch number if needed)
checkpoint_path = os.path.join("checkpoints", f"{MODEL_NAME}_epoch_10.pth")

model.load_state_dict(torch.load(checkpoint_path, map_location=device, weights_only=True))
model.to(device)

model.eval()

print(f"Successfully loaded {MODEL_NAME} weights from {checkpoint_path}")
print("Model is ready for inference or testing!")

ModuleNotFoundError: No module named 'config'